# Continuous Wavelet Transform (CWT) for R-Peak Detection
## Proper Research-Grade Implementation on MIT-BIH Arrhythmia Database
In this notebook, we evaluate a Continuous Wavelet Transform (CWT) based R-peak detection algorithm across the MIT-BIH dataset. 
We use the **Mexican Hat (`mexh`)** wavelet at an optimal scale corresponding to typical QRS complex frequencies (~15 Hz).

We extract annotations, detect peaks using adaptive thresholding on the CWT energy, match them against ground truth, and generate summary metrics:
*   **Sensitivity (Se)**
*   **Positive Predictivity (+P)**
*   **Error Rate**



In [ ]:
import os
import glob
import wfdb
import pywt
import numpy as np
import scipy.signal as signal
import pandas as pd
import matplotlib.pyplot as plt
import time

plt.rcParams['figure.figsize'] = (12, 4)


## 1. Helper Functions for Annotation and Evaluation


In [ ]:
def get_annotated_peaks(record_path):
    """
    Reads the MIT-BIH annotation file (.atr) and extracts only true beat annotations.
    """
    ann = wfdb.rdann(record_path, 'atr')
    
    # Standard MIT-BIH beat types (excluding non-beat symbols like '+', '~', 'x')
    beat_types = ['N', 'L', 'R', 'B', 'A', 'a', 'J', 'S', 'V', 'r', 'F', 'e', 'j', 'n', 'E', '/', 'f', 'Q', '?']
    
    beats = [ann.sample[i] for i in range(len(ann.symbol)) if ann.symbol[i] in beat_types]
    return np.array(beats)

def evaluate_peaks(detected_peaks, annotated_peaks, fs, tolerance_ms=150):
    """
    Matches detected peaks with annotated peaks within a tolerance window (default 150ms).
    """
    tolerance = int(tolerance_ms / 1000.0 * fs)
    
    tp = 0
    fn = 0
    
    detected_matched = set()
    
    for ann in annotated_peaks:
        close_peaks = [p for p in detected_peaks if abs(p - ann) <= tolerance]
        if close_peaks:
            # Pick the closest one
            closest = min(close_peaks, key=lambda x: abs(x - ann))
            if closest not in detected_matched:
                tp += 1
                detected_matched.add(closest)
            else:
                fn += 1
        else:
            fn += 1
            
    fp = len(detected_peaks) - len(detected_matched)
    
    return tp, fp, fn



## 2. CWT Peak Detection Algorithm


In [ ]:
def detect_r_peaks_cwt(ecg, fs):
    """
    Detects R-peaks using Mexican Hat Wavelet Transform and an adaptive threshold.
    """
    # Scale selection for mexh wavelet to target 10-25 Hz QRS energy.
    # Center frequency of mexh is 0.25. (360 * 0.25) / freq => approx scales [4, 5, 6]
    scales = np.array([4, 5, 6])
    
    # 1. Compute CWT
    cwt_coeffs, _ = pywt.cwt(ecg, scales, 'mexh')
    
    # 2. Sum of squared coefficients (Energy)
    energy = np.sum(cwt_coeffs ** 2, axis=0)
    
    # 3. Smoothing filter (Moving average)
    window_size = int(0.10 * fs)
    smoothed = np.convolve(energy, np.ones(window_size) / window_size, mode='same')
    
    # 4. Local Maxima Candidate Selection
    min_distance = int(0.20 * fs) # minimum 200ms between peaks
    candidate_peaks, _ = signal.find_peaks(smoothed, distance=min_distance)
    
    # 5. Adaptive Thresholding (similar to Pan-Tompkins style)
    if len(smoothed) > 2*fs:
        spk = np.max(smoothed[:2*fs])
        npk = np.mean(smoothed[:2*fs])
    else:
        spk = np.max(smoothed)
        npk = np.mean(smoothed)
        
    threshold1 = npk + 0.25 * (spk - npk)
    
    r_peaks = []
    
    for peak in candidate_peaks:
        peak_val = smoothed[peak]
        if peak_val > threshold1:
            spk = 0.125 * peak_val + 0.875 * spk
            r_peaks.append(peak)
        else:
            npk = 0.125 * peak_val + 0.875 * npk
            
        threshold1 = npk + 0.25 * (spk - npk)
        
    # 6. Refine peak locations against original ECG to avoid phase shift delays
    search_radius = int(0.10 * fs)
    refined_peaks = []
    for p in r_peaks:
        start = max(0, p - search_radius)
        end = min(len(ecg), p + search_radius)
        if start < end:
            # We look for the maximum absolute value in the original signal near the peak
            # Since QRS can be positive or negative, argmax of absolute is more robust
            local_max = np.argmax(np.abs(ecg[start:end]))
            refined_peaks.append(start + local_max)
            
    return np.array(sorted(set(refined_peaks)))



## 3. Visual Example (First 10 seconds of Record 100)


In [ ]:
record_path = '100'
record = wfdb.rdsamp(record_path, sampto=3600)
ecg = record[0][:, 0]
fs = record[1]['fs']
ann_peaks = get_annotated_peaks(record_path)
ann_peaks = ann_peaks[ann_peaks < 3600]

det_peaks = detect_r_peaks_cwt(ecg, fs)

t = np.arange(len(ecg)) / fs
plt.plot(t, ecg, label='ECG Signal')
plt.scatter(t[ann_peaks], ecg[ann_peaks], marker='o', color='green', s=100, facecolors='none', label='Annotated Peaks')
plt.scatter(t[det_peaks], ecg[det_peaks], marker='x', color='red', s=80, label='CWT Detected Peaks')
plt.title(f"CWT Detection on Record {record_path}")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend(loc='upper right')
plt.grid(True)
plt.show()



## 4. Full Dataset Evaluation
We will now iterate through all available `.dat` files (MIT-BIH records) in the directory, running the detector over the full 30-minute signals.



In [ ]:
records = [os.path.basename(f).split('.')[0] for f in glob.glob('*.dat')]
records.sort()

results = []

print(f"Starting evaluation on {len(records)} records...")
start_time = time.time()

for rec in records:
    try:
        record_path = rec
        record = wfdb.rdsamp(record_path)
        ecg = record[0][:, 0] # channel 0
        fs = record[1]['fs']
        
        ann_peaks = get_annotated_peaks(record_path)
        
        t0 = time.time()
        det_peaks = detect_r_peaks_cwt(ecg, fs)
        proc_time = time.time() - t0
        
        tp, fp, fn = evaluate_peaks(det_peaks, ann_peaks, fs)
        
        results.append({
            'Record': rec,
            'Total Beats': len(ann_peaks),
            'TP': tp,
            'FP': fp,
            'FN': fn,
            'Time (s)': proc_time
        })
    except Exception as e:
        print(f"Error processing {rec}: {e}")

eval_time = time.time() - start_time
print(f"\nEvaluation completed in {eval_time:.2f} seconds.")



## 5. Summary Metrics


In [ ]:
df = pd.DataFrame(results)

# Calculate metrics per record
df['Sensitivity (%)'] = (df['TP'] / (df['TP'] + df['FN']) * 100).fillna(0)
df['Predictivity (%)'] = (df['TP'] / (df['TP'] + df['FP']) * 100).fillna(0)
df['Error Rate (%)'] = ((df['FP'] + df['FN']) / df['Total Beats'] * 100).fillna(0)

# Overall totals
total_beats = df['Total Beats'].sum()
total_tp = df['TP'].sum()
total_fp = df['FP'].sum()
total_fn = df['FN'].sum()

overall_se = total_tp / (total_tp + total_fn) * 100
overall_pp = total_tp / (total_tp + total_fp) * 100
overall_er = (total_fp + total_fn) / total_beats * 100

print("="*50)
print("OVERALL PERFORMANCE")
print("="*50)
print(f"Total Beats Evaluated : {total_beats}")
print(f"True Positives (TP)   : {total_tp}")
print(f"False Positives (FP)  : {total_fp}")
print(f"False Negatives (FN)  : {total_fn}")
print(f"Sensitivity (Se)      : {overall_se:.2f} %")
print(f"Positive Predictivity : {overall_pp:.2f} %")
print(f"Detection Error Rate  : {overall_er:.2f} %")
print("="*50)

# Display table
display(df.round(2))



In [ ]:
# Save the results to CSV for comparison with other methods
df.to_csv('cwt_mitbih_results.csv', index=False)
print("Results saved to 'cwt_mitbih_results.csv'")

